In [41]:
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
from tqdm import tqdm
from re import sub
import re
tqdm.pandas()

In [14]:
word_vectors = Word2Vec.load("./sentiment140-word2vec.model").wv

In [31]:
df = pd.read_csv("./sentiment140-cleaned.csv", names=['sentiment', 'id', 'date', 'flag', 'user',  'tweet', 'text'], encoding = "ISO-8859-1", engine="python")

In [39]:
def cleantext(text):
    text = sub(r'\w+:\/{2}[\d\w-]+(\.[\d\w-]+)*(?:(?:\/[^\s/]*))*', '', text, flags=re.MULTILINE)
    text = text.lower()
    text = sub(r'[^A-Za-z0-9!?#@]', ' ', text)
    text = sub(r'\?+', ' ? ', text)
    text = sub(r'\!+', ' ! ', text)

    text = sub(r'\s+', ' ', text)
    return text.split()

In [42]:
df['text'] = df.progress_apply(lambda row: cleantext(row['tweet']), axis=1)

100%|██████████| 1600000/1600000 [00:50<00:00, 31710.85it/s]


In [33]:
df.groupby('sentiment').nth(0)

,id,date,user,flag,tweet,text
sentiment,,,,,,
0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...","['@switchfoot', 'awww', 'that', 's', 'a', 'bum..."
4,1467822272,Mon Apr 06 22:22:45 PDT 2009,NO_QUERY,ersle,I LOVE @Health4UandPets u guys r the best!!,"['i', 'love', '@health4uandpets', 'u', 'guys',..."


In [15]:
cluster_good = ["good", "nice", "cool", "lovely", "wonderful", "great", "awesome", "fantastic", "amazing", "fun", "excellent"]
cluster_bad = ["bad", "horrible", "terrible", "awful", "worst", "shitty", "crappy", "sucks", "hate"]

In [19]:
def clusterDistance(word, cluster):
    return np.average(word_vectors.distances(word, cluster))

In [26]:
print(clusterDistance("good", cluster_good))
print(clusterDistance("bad", cluster_good))

print(clusterDistance("bad", cluster_bad))
print(clusterDistance("good", cluster_bad))

print(clusterDistance("i", cluster_good))
print(clusterDistance("i", cluster_bad))

0.5205891
0.8142739
0.49157986
0.70155805
0.87830466
0.7944416


In [27]:
cutoff = 0.6

In [45]:
def tweetClusterDistance(text, cluster):
    sum = 0
    for word in text:
        if not (word in word_vectors.vocab):
            continue
        if clusterDistance(word, cluster) < cutoff:
            sum += 1
    return sum

In [46]:
df['positive'] = df.progress_apply(lambda row: tweetClusterDistance(row['text'], cluster_good), axis=1)

100%|██████████| 1600000/1600000 [18:02<00:00, 1477.52it/s]


In [47]:
df['negative'] = df.progress_apply(lambda row: tweetClusterDistance(row['text'], cluster_bad), axis=1)

100%|██████████| 1600000/1600000 [18:08<00:00, 1469.86it/s]


In [48]:
df.to_csv("./sentiment140-sentimented.csv", index=False)

In [49]:
def getSentiment(row):
    if row['positive'] > row['negative']:
        return 4
    return 0

In [50]:
df['predict'] = df.progress_apply(lambda row: getSentiment(row), axis=1)

100%|██████████| 1600000/1600000 [00:18<00:00, 85326.63it/s] 


In [51]:
df['predict_correct'] = df.progress_apply(lambda row: row['sentiment'] == row['predict'], axis=1)

100%|██████████| 1600000/1600000 [00:16<00:00, 99023.02it/s] 


In [55]:
df['predict_correct'].value_counts()

True     882534
False    717466
Name: predict_correct, dtype: int64

In [54]:
print(882534/1600000)

0.55158375
